# Semana 10
## Series temporales, rolling windows y comparaciones interperiodo

**Objetivo**: analizar datos en el tiempo con agregación correcta, suavizado y comparación entre periodos.

**Herramientas teoricas de la semana**
- tendencia y estacionalidad
- granularidad temporal
- moving average
- comparacion interperiodo


### Agenda sugerida de 4 horas
- 0:00 - 0:30: teoria temporal y riesgos de interpretacion
- 0:30 - 1:20: agregacion mensual y KPIs temporales
- 1:20 - 2:10: rolling average y variacion relativa
- 2:10 - 3:20: vista temporal explicativa y exportables
- 3:20 - 4:00: discusion metodologica


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-10"
OUTPUT_DIR = ensure_output_dir(WEEK)

clean, _ = clean_sales_data(introduce_quality_issues(make_base_sales(n=2600, seed=29), seed=29))
monthly = (
    clean.assign(month=clean['order_date'].dt.to_period('M').dt.to_timestamp())
    .groupby('month', as_index=False)
    .agg(monthly_sales=('sales', 'sum'), monthly_profit=('profit', 'sum'))
    .sort_values('month')
)
monthly['rolling_3m_sales'] = monthly['monthly_sales'].rolling(3).mean()
monthly['pct_change_mom'] = monthly['monthly_sales'].pct_change()
monthly.head()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(monthly['month'], monthly['monthly_sales'], label='monthly_sales', color='#111827')
ax.plot(monthly['month'], monthly['rolling_3m_sales'], label='rolling_3m_sales', color='#9ca3af', linewidth=3)
ax.set_title('Venta mensual y promedio movil de 3 meses')
ax.legend()
plt.tight_layout()


In [ ]:
monthly['year'] = monthly['month'].dt.year
monthly['month_num'] = monthly['month'].dt.month
yoy = monthly.pivot(index='month_num', columns='year', values='monthly_sales')
yoy


In [ ]:
insight_table = pd.DataFrame([
    {'question': '¿Hay una tendencia sostenida?', 'evidence': 'Comparar monthly_sales con rolling_3m_sales'},
    {'question': '¿Hay meses especialmente atipicos?', 'evidence': 'Revisar pct_change_mom y picos en la serie'},
    {'question': '¿La forma se repite entre años?', 'evidence': 'Comparar tabla YoY por month_num'},
])
insight_table


In [ ]:
save_for_tableau(monthly, WEEK, 'monthly_metrics')
save_for_tableau(yoy.reset_index(), WEEK, 'yoy_monthly_sales')
save_for_tableau(insight_table, WEEK, 'temporal_questions')


### Uso teorico de herramientas
- `rolling()` implementa la idea de suavizado.
- `pct_change()` permite hablar de variación relativa y no solo de volumen.
- El pivote anual facilita comparar estacionalidad sin depender exclusivamente del gráfico.
